# 深层特工额外：幕后花絮

201 教您如何**使用**深度代理。本笔记本将教您它们如何**工作**以及如何定制重要的部件。

**您将学到什么：**
- `create_deep_agent()` 实际上只是 `create_agent()` + 中间件
- 如何构建自己的文件系统接口（模式，而不是完整的实现）
- 如何用结构化任务定义替换默认的 `task()` 工具
- 如何将确定性 `StateGraph` 工作流程作为子代理插入
- MCP 适配器如何使外部工具与本机工具互换
- 如何使用沙箱执行代码
- 上下文管理失败时会发生什么以及如何修复它

**先决条件：** 首先完成 201 Deep Agents 笔记本。

## 第 0 部分：设置

In [1]:
# Add project root to Python path
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model

print(f"Model: {model.__class__.__name__}")

Model: ChatOpenAI


In [2]:
# 笔记本共享导入
from langchain_core.tools import tool
from langsmith import uuid7
from langgraph.checkpoint.memory import MemorySaver

from tavily import TavilyClient

tavily_client = TavilyClient()


@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.

    Args:
        query: Search query to execute.
    """
    results = tavily_client.search(query, max_results=3)
    return "\n\n".join(
        f"**{r['title']}**\n{r['url']}\n{r['content']}" for r in results["results"]
    )

---
## 第 1 部分：解构 `create_deep_agent` 

 `create_deep_agent()` 是 `create_agent()` 的包装器，用于组装特定的中间件堆栈。

这是它的作用（简化）：

 ```python
def create_deep_agent(model, tools, system_prompt, ...):
    middleware = [
        TodoListMiddleware(),                    # Planning
        FilesystemMiddleware(backend=backend),   # File tools
        SubAgentMiddleware(subagents=...),       # task() tool
        SummarizationMiddleware(model, ...),     # Context management
        PatchToolCallsMiddleware(),              # Fix dangling tool calls
    ]
    return create_agent(model, tools=tools, middleware=middleware)
``` 

### 为什么这很重要

如果您可以自己构建中间件堆栈，您可以：
- 交换任何部分（例如，您自己的文件系统、您自己的任务工具）
- 以不同的顺序添加中间件
- 直接使用 `create_agent()` ，无需深度代理依赖

In [3]:
# 使用 create_agent 从头构建同样的能力
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.summarization import create_summarization_middleware
from deepagents.middleware.patch_tool_calls import PatchToolCallsMiddleware
from deepagents.backends import StateBackend

# This is the core of what create_deep_agent does
hand_built_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt='你是一个乐于助人的研究助手。',
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        create_summarization_middleware(model, StateBackend),
        PatchToolCallsMiddleware(),
    ],
).with_config({"recursion_limit": 1000})

print('Agent built with create_agent + middleware')
print(f"Type: {type(hand_built_agent).__name__}")

Agent built with create_agent + middleware
Type: CompiledStateGraph


In [4]:
# Test it -- it behaves exactly like create_deep_agent
config = {"configurable": {"thread_id": str(uuid7())}}

result = hand_built_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "写入文件 /notes.md，内容为“从头构建！”，然后读回该文件。",
            }
        ]
    },
    config=config,
)

print(result["messages"][-1].content)
print('\nFiles in state:', list(result.get("files", {}).keys()))

I have written the file /notes.md with the content "Built from scratch!" and read it back for you. The content of the file is:

Built from scratch!

Files in state: ['/notes.md']


### 中间件钩子一览

每个中间件都可以实现这些钩子：

|钩|当它运行时 |常用|
|------|-------------|------------|
|  `before_agent(state, runtime)` |开始时一次 |加载技能、记忆|
|  `wrap_model_call(request, handler)` |每个法学硕士电话 |注入系统提示文字|
|  `wrap_tool_call(request, handler)` |每个工具调用|记录、拦截、修改|
|  `tools`（属性）|报名时间 |向代理添加工具 |
|  `state_schema`（属性）|图表编译|使用新字段扩展状态 |

让我们通过一个简单的日志记录中间件来看看这些钩子的作用：

In [5]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call


@wrap_model_call
def inspect_model_call(request, handler):
    """See what the LLM receives on each call."""
    # request.system_message contains the full system prompt (all middleware have appended to it)
    sys_msg = request.system_message
    if sys_msg:
        # System message can be multi-block from multiple middleware
        total_chars = sum(
            len(b.get("text", ""))
            for b in sys_msg.content_blocks
            if isinstance(b, dict)
        )
        print(
            f"\n🧠 [Model Call] System prompt: {total_chars:,} chars across {len(sys_msg.content_blocks)} blocks"
        )
    print(f"   Tools available: {[t.name for t in request.tools]}")
    print(f"   Messages in context: {len(request.messages)}")
    return handler(request)


@wrap_tool_call
def inspect_tool_call(request, handler):
    """See what tools are called and with what args."""
    name = request.tool_call["name"]
    args = request.tool_call["args"]
    # Truncate long args for display
    display_args = {
        k: (v[:80] + "..." if isinstance(v, str) and len(v) > 80 else v)
        for k, v in args.items()
    }
    print(f"   🔧 {name}({display_args})")
    return handler(request)

    # Add our inspect middleware to the stack


inspectable_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt='你是一个乐于助人的助手。',
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        create_summarization_middleware(model, StateBackend),
        PatchToolCallsMiddleware(),
        inspect_model_call,  # Our custom middleware
        inspect_tool_call,
    ],
).with_config({"recursion_limit": 1000})

config = {"configurable": {"thread_id": str(uuid7())}}
result = inspectable_agent.invoke(
    {"messages": [{"role": "user", "content": "写入文件 /hello.txt，内容为“hi”"}]},
    config=config,
)

print("\n---")
print(result["messages"][-1].content)


🧠 [Model Call] System prompt: 2,272 chars across 3 blocks
   Tools available: ['write_todos', 'ls', 'read_file', 'write_file', 'edit_file', 'glob', 'grep', 'tavily_search']
   Messages in context: 1
   🔧 write_file({'file_path': '/hello.txt', 'content': 'hi'})

🧠 [Model Call] System prompt: 2,272 chars across 3 blocks
   Tools available: ['write_todos', 'ls', 'read_file', 'write_file', 'edit_file', 'glob', 'grep', 'tavily_search']
   Messages in context: 3

---
I have written the file /hello.txt with the content 'hi'.


### 要点
- `create_deep_agent()` = `create_agent()` + 特定的中间件堆栈
- 中间件挂钩让您可以检查、修改和扩展代理循环的任何部分
- 您可以交换、重新排序或替换任何中间件来自定义行为

---
## 第 2 部分：构建最小文件系统接口

Deep Agent 的 `FilesystemMiddleware` 提供了 7 个工具（ `ls` 、 `read_file` 、 `write_file` 、 `edit_file` 、 `glob` 、 `grep` 、 `execute` ）。让我们构建一个最小的 2 工具版本来理解 **模式**，而不是重新实现所有这些。

**文件系统工具不直接接触文件系统**。他们：
1.通过后端抽象进行操作（`BackendProtocol`）
2.返回`Command(update={"files": {...}})`来更新LangGraph状态
3. 使用自定义状态缩减器来合并文件更新

此模式将文件存储的“位置”与代理使用它们的“方式”分开。

In [6]:
from datetime import datetime, timezone
from typing import Annotated, Any, NotRequired
from langchain.agents.middleware.types import AgentMiddleware, AgentState
from langchain.tools import ToolRuntime
from langchain_core.messages import ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.types import Command
from langchain_core.messages import SystemMessage


# 第 1 步：定义状态 reducer
# LangGraph 会用它合并多次工具调用产生的文件更新
def file_reducer(left: dict | None, right: dict) -> dict:
    """Merge file state updates. None values = deletions."""
    result = {**(left or {})}
    for key, value in right.items():
        if value is None:
            result.pop(key, None)  # 删除
        else:
            result[key] = value  # 创建/更新
    return result

    # 第 2 步：定义状态 schema


class MinimalFilesystemState(AgentState):
    files: Annotated[NotRequired[dict], file_reducer]

    # 第 3 步：构建中间件


class MinimalFilesystemMiddleware(AgentMiddleware):
    """A stripped-down filesystem middleware to show the pattern."""

    state_schema = MinimalFilesystemState

    def __init__(self):
        super().__init__()
        self.tools = [self._make_write_tool(), self._make_read_tool()]

    def _make_write_tool(self):
        def write_file(
            file_path: Annotated[str, '文件的绝对路径（例如 /notes.md）'],
            content: Annotated[str, '要写入的内容'],
            runtime: ToolRuntime,
        ) -> Command:
            """向虚拟文件系统写入文件。"""
            now = datetime.now(timezone.utc).isoformat()
            file_data = {
                "content": content.split("\n"),
                "created_at": now,
                "modified_at": now,
            }
            # 这是关键模式：返回 Command 来更新状态
            return Command(
                update={
                    "files": {file_path: file_data},
                    "messages": [
                        ToolMessage(
                            f"Wrote {len(content)} chars to {file_path}",
                            tool_call_id=runtime.tool_call_id,
                        )
                    ],
                }
            )

        return StructuredTool.from_function(name="write_file", func=write_file)

    def _make_read_tool(self):
        def read_file(
            file_path: Annotated[str, 'Absolute path of the file to read'],
            runtime: ToolRuntime,
        ) -> Command:
            """Read a file from the virtual filesystem."""
            files = runtime.state.get("files", {})
            if file_path not in files:
                return Command(
                    update={
                        "messages": [
                            ToolMessage(
                                f"Error: {file_path} not found",
                                tool_call_id=runtime.tool_call_id,
                            )
                        ],
                    }
                )
            content = "\n".join(files[file_path]["content"])
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content,
                            tool_call_id=runtime.tool_call_id,
                        )
                    ],
                }
            )

        return StructuredTool.from_function(name="read_file", func=read_file)

    def wrap_model_call(self, request, handler):
        """Tell the LLM about its filesystem capabilities."""
        instructions = """## Filesystem
You have  `write_file`  and  `read_file`  tools. Use absolute paths (e.g.,  `/notes.md` ).
Files are stored in-memory and persist within this thread."""
        # Append our instructions to the existing system message
        existing = request.system_message
        blocks = list(existing.content_blocks) if existing else []
        blocks.append({"type": "text", "text": f"\n\n{instructions}"})
        new_sys = SystemMessage(content_blocks=blocks)
        return handler(request.override(system_message=new_sys))


print('MinimalFilesystemMiddleware defined')
print("Tools:", [t.name for t in MinimalFilesystemMiddleware().tools])

MinimalFilesystemMiddleware defined
Tools: ['write_file', 'read_file']


In [7]:
# 使用 create_agent 搭配我们的最小文件系统
minimal_fs_agent = create_agent(
    model,
    system_prompt='你是一个乐于助人的助手。',
    middleware=[MinimalFilesystemMiddleware()],
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = minimal_fs_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "写入文件 /demo.txt，内容为“来自最小文件系统的你好！”，然后读回该文件。",
            }
        ]
    },
    config=config,
)

print('智能体响应:', result["messages"][-1].content)
print(
    '\nFiles in state:', {k: v["content"] for k, v in result.get("files", {}).items()}
)

Agent response: I have written "Hello from minimal filesystem!" to the file /demo.txt and read it back. The content is:
Hello from minimal filesystem!

Files in state: {'/demo.txt': ['Hello from minimal filesystem!']}


### 这与真实的 `FilesystemMiddleware` 相比如何 

我们的最小版本显示了核心模式。真正的实现添加了：

|特色|我们的版本|真实 `FilesystemMiddleware` |
|--------|-------------|----------------------------|
|工具|  `write_file` , `read_file` | + `ls` 、 `edit_file` 、 `glob` 、 `grep` 、 `execute` |
|存储|直接状态访问 |  `BackendProtocol` 抽象 |
|分页|没有 |  `offset` / `limit` 读取 |
|图片 |没有 |多模式内容块 |
|沙盒|没有 |  `SandboxBackendProtocol` |
|路径验证|没有 |清理、虚拟路径 |

### 要点
- 文件系统工具通过 `Command(update={"files": {...}})` 修改状态，而不是通过写入磁盘
- 自定义状态减速器（`file_reducer`）处理合并和删除
- `wrap_model_call` 注入指令，以便法学硕士知道哪些工具可用
- 真正的实现添加了后端抽象，因此存储是可插拔的

---
## 第 3 部分：后端实现 — 文件实际存在的位置

在第 2 部分中，我们构建了一个以 LangGraph 状态存储文件的文件系统。但这只是一种选择。 Deep Agents 具有三种后端实现，每种都具有根本不同的存储语义。

**工具不知道文件存储在哪里**。他们调用 `backend.write()` ，后端决定是否更新状态、写入磁盘或调用远程存储。

### `WriteResult` 分支模式

每个后端的 `write()` 返回一个 `WriteResult` ，其中包含三个字段：
 ```python
@dataclass
class WriteResult:
    error: str | None = None        # Error message on failure
    path: str | None = None          # Path of written file
    files_update: dict | None = None  # State update dict (or None)
``` 

`files_update` 字段控制分支：
- ** `StateBackend` **：返回 `files_update={path: file_data}` → 中间件将其包装在 `Command(update={"files": ...})` 中 
- ** `FilesystemBackend` **：返回`files_update=None` → 文件已在磁盘上，中间件仅返回字符串确认
- ** `StoreBackend` **：返回 `files_update=None` → 文件已在商店中，无需更新状态

 `FilesystemMiddleware._create_write_file_tool()`：
 ```python
res: WriteResult = backend.write(file_path, content)
if res.files_update is not None:
    # StateBackend: update LangGraph state
    return Command(update={"files": res.files_update, "messages": [...]})
# FilesystemBackend/StoreBackend: already persisted
return f"Updated file {res.path}"
```

In [8]:
# 让我们在实践中看看差异
from deepagents.backends.protocol import WriteResult

# Simulate what each backend returns for the same write() call

# StateBackend: "here's the data, put it in state"
state_result = WriteResult(
    path='/notes.md',
    files_update={
        '/notes.md': {"content": ["hello"], "created_at": "...", "modified_at": "..."}
    },
)
print(
    f"StateBackend:      files_update = {type(state_result.files_update).__name__} (middleware issues Command)"
)

# FilesystemBackend: "I already wrote it to disk"
fs_result = WriteResult(path='/notes.md', files_update=None)
print(
    f"FilesystemBackend: files_update = {fs_result.files_update} (middleware returns string)"
)

# StoreBackend: "I already put it in the Store"
store_result = WriteResult(path='/notes.md', files_update=None)
print(
    f"StoreBackend:      files_update = {store_result.files_update} (middleware returns string)"
)

StateBackend:      files_update = dict (middleware issues Command)
FilesystemBackend: files_update = None (middleware returns string)
StoreBackend:      files_update = None (middleware returns string)


### StateBackend — 短暂的，处于 LangGraph 状态

这是默认设置。文件位于 `runtime.state["files"]` — 一个由 LangGraph 检查点管理的字典中。

 ```python
class StateBackend(BackendProtocol):
    def __init__(self, runtime: ToolRuntime):
        self.runtime = runtime
    
    def read(self, file_path, ...):
        files = self.runtime.state.get("files", {})
        return files[file_path]  # Read directly from state
    
    def write(self, file_path, content):
        file_data = create_file_data(content)
        # Return the data so middleware can update state via Command
        return WriteResult(path=file_path, files_update={file_path: file_data})
``` 

**生命周期：** 文件在线程内持续存在（通过检查指针），但在线程之间消失。

**权衡：**快速（内存中），但每个文件都会序列化到检查点中。大文件会使检查点大小膨胀。

In [9]:
# StateBackend in action — files live in state["files"]
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

state_agent = create_deep_agent(
    model=model,
    backend=StateBackend(),  # Default — pass an instance
    system_prompt='你是一个乐于助人的助手。',
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = state_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "写入文件 /demo.txt，内容为“来自 StateBackend 的你好”",
            }
        ]
    },
    config=config,
)

# Files are in the result state
print('Files in state:', list(result.get("files", {}).keys()))
if '/demo.txt' in result.get("files", {}):
    print('Content:', result["files"]['/demo.txt']["content"])

Files in state: ['/demo.txt']
Content: ['hello from StateBackend']


### FilesystemBackend — 真实磁盘 I/O

读取和写入磁盘上的实际文件。关键参数是 `virtual_mode` ：

- `virtual_mode=True` ：路径在 `root_dir` 下沙箱化。代理写入 `/notes.md` → 实际上写入 `{root_dir}/notes.md` 。路径遍历 ( `..` ) 被阻止。
- `virtual_mode=False`（默认，已弃用）：绝对路径按原样工作。 **没有安全边界。**

 ```python
class FilesystemBackend(BackendProtocol):
    def __init__(self, root_dir, virtual_mode=True):
        self.cwd = Path(root_dir).resolve()
        self.virtual_mode = virtual_mode
    
    def write(self, file_path, content):
        resolved = self._resolve_path(file_path)  # Sandboxing happens here
        resolved.parent.mkdir(parents=True, exist_ok=True)
        resolved.write_text(content)
        return WriteResult(path=file_path, files_update=None)  # Already on disk!
``` 

**与 StateBackend 的主要区别：** `files_update=None` — 文件已保存到磁盘。中间件返回一个纯字符串响应，不需要 `Command` 。

**生命周期：** 文件永久保留在磁盘上。持久性不需要检查指针，但文件在使用相同 `root_dir` 的所有线程之间共享。

In [10]:
# FilesystemBackend in action — files go to real disk
import tempfile, os
from deepagents.backends import FilesystemBackend

sandbox_dir = tempfile.mkdtemp(prefix="deepagents_extra_")
print(f"Sandbox directory: {sandbox_dir}")

fs_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True),
    system_prompt='你是一个乐于助人的助手。',
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = fs_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "写入文件 /demo.txt，内容为“来自 FilesystemBackend 的你好”",
            }
        ]
    },
    config=config,
)

# No files in state — they're on disk!
print(f"Files in state: {list(result.get('files', {}).keys())}")

# But they exist on disk
disk_path = os.path.join(sandbox_dir, 'demo.txt')
if os.path.exists(disk_path):
    print(f"File on disk: {open(disk_path).read()}")

    # Cleanup
import shutil

shutil.rmtree(sandbox_dir, ignore_errors=True)

Sandbox directory: /var/folders/jj/2fvdkyfj0856p6_6sdvv74rw0000gn/T/deepagents_extra_w96hwg1a
Files in state: []
File on disk: hello from FilesystemBackend


### StoreBackend — 持久、跨线程存储

使用 LangGraph 的 `BaseStore` （键值存储）。文件在所有线程中持久保存 - 在线程 A 中写入，在线程 B 中读取。

 ```python
class StoreBackend(BackendProtocol):
    def __init__(self, runtime: ToolRuntime, *, namespace=None):
        self.runtime = runtime
    
    def write(self, file_path, content):
        store = self.runtime.store          # LangGraph provides this
        namespace = self._get_namespace()    # e.g., ("filesystem",)
        file_data = create_file_data(content)
        store.put(namespace, file_path, file_data)  # Persisted immediately
        return WriteResult(path=file_path, files_update=None)  # Already in store!
``` 

**命名空间范围**控制隔离：
- `("filesystem",)` — 所有线程/用户共享文件
- `(user_id, "filesystem")` — 每用户隔离
- `(assistant_id, "filesystem")` — 每个助理隔离

**生命周期：** 只要存储存在，文件就会持续存在。在 LangGraph Platform 中，这意味着跨部署。

In [11]:
# StoreBackend in action — files persist across threads
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

store_agent = create_deep_agent(
    model=model,
    backend=StoreBackend(namespace=lambda rt: ("filesystem",)),
    store=store,
    system_prompt='你是一个乐于助人的助手。',
    checkpointer=MemorySaver(),
)

# 线程 1：写入文件
config_t1 = {"configurable": {"thread_id": str(uuid7())}}
result = store_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "写入文件 /shared_note.txt，内容为“这会跨线程持久保存！”",
            }
        ]
    },
    config=config_t1,
)
print('Thread 1:', result["messages"][-1].content)

# Thread 2: Read it (different thread!)
config_t2 = {"configurable": {"thread_id": str(uuid7())}}
result = store_agent.invoke(
    {"messages": [{"role": "user", "content": 'Read the file /shared_note.txt'}]},
    config=config_t2,
)
print('Thread 2:', result["messages"][-1].content)

Thread 1: File /shared_note.txt has been written with the content: 'This persists across threads!'
Thread 2: The content of the file /shared_note.txt is:
"This persists across threads!"


### 后端比较

| |状态后端 |文件系统后端 |商店后台 |
|---|---|---|---|
| **存储** | LangGraph 状态（检查点）|真实文件系统 | LangGraph 商店（键值）|
| **坚持** |线程内 |永久保存在磁盘上 |跨线程、跨会话|
| ** `files_update` ** |  `{path: data}` → `Command` |  `None` → 字符串 |  `None` → 字符串 |
| **国家膨胀** |是（检查点中的文件）|没有 |没有 |
| **用例** |暂存空间，笔记本| CLI 工具、本地开发 | LangGraph 平台，生产 |
| **需要 `store=` ** |没有 |没有 |是（`InMemoryStore` 或平台商店）|

在生产中，您通常使用 `CompositeBackend` 来混合它们 - 通过 `StateBackend` 提供临时暂存空间，通过 `StoreBackend` 提供持久内存：

 ```python
composite_backend = CompositeBackend(
    default=StateBackend(),           # Scratch files go to state
    routes={
        "/memories/": StoreBackend(),  # Memories persist in Store
    },
)
``` 

### 要点
- `WriteResult.files_update` 字段控制中间件是否发出 `Command` 还是返回字符串
- `StateBackend` 返回状态更新数据；  `FilesystemBackend` 和 `StoreBackend` 直接持久化并返回 `None` 
- 所有三个后端都实现相同的 `BackendProtocol` — 工具不知道它们正在使用哪一个
- `CompositeBackend` 将不同路径路由到不同后端以进行混合存储

---
## 第 4 部分：子代理的自定义任务定义

默认情况下，`SubAgentMiddleware` 创建一个带有两个参数的 `task()` 工具：
- `description` (str) — 描述要做什么的原始字符串
- `subagent_type` (str) — 要调用哪个子代理

子代理接收 `HumanMessage(content=description)` — 只是一个扁平字符串。

这可行，但有局限性：
- 没有结构——法学硕士必须将所有内容打包成散文
- 没有类型安全——没有对通过的内容进行验证
- 无选择性上下文 - 子代理获取所有非排除状态

让我们构建自定义任务工具来解决这些问题。

In [12]:
# The default task tool flow (simplified):
# 1. Main agent calls: task(description="Research LangGraph", subagent_type="researcher")
# 2. SubAgentMiddleware:
# a. Filters state: removes internal keys (messages, todos, etc.) so subagents get clean context
# b. Sets messages: [HumanMessage("Research LangGraph")]  <-- just the raw string
# c. Invokes subagent
# d. Returns: Command(update= {"messages": [ToolMessage(result)]} )
#
# The limitation: "description" is a raw string. The LLM has to pack
# everything — topic, depth, format, constraints — into prose.
# 我们用结构化输入来解决这个问题。

In [13]:
# Custom Task Definition 1: Structured Input
#
# Instead of a raw string, define a Pydantic model for the task input.
# This gives the orchestrator agent a clear schema to fill out.

from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage


class ResearchTask(BaseModel):
    """A structured research task definition."""

    topic: str = Field(description='The topic to research')
    depth: str = Field(
        description="'shallow' (quick overview), 'medium' (key details), or 'deep' (comprehensive)"
    )
    output_format: str = Field(
        description="'bullet_points', 'narrative', or 'structured_report'"
    )
    max_sources: int = Field(
        default=3, description='Maximum number of sources to consult'
    )
    focus_areas: list[str] = Field(
        default_factory=list, description='Specific aspects to focus on'
    )


def build_structured_task_tool(subagent_graph):
    """Build a task tool with structured input instead of a raw string."""

    def research(
        topic: Annotated[str, 'The topic to research'],
        depth: Annotated[str, "'shallow', 'medium', or 'deep'"],
        output_format: Annotated[
            str, "'bullet_points', 'narrative', or 'structured_report'"
        ],
        max_sources: Annotated[int, 'Maximum sources to consult'] = 3,
        focus_areas: Annotated[list[str], 'Specific aspects to focus on'] = [],
        runtime: ToolRuntime = None,
    ) -> Command:
        """Delegate a structured research task to the research subagent."""
        # Build a well-structured prompt from the typed fields
        focus_str = (
            "\n".join(f"  - {area}" for area in focus_areas)
            if focus_areas
            else '  (none specified)'
        )

        prompt = f"""## Research Task

**Topic:** {topic}
**Depth:** {depth}
**Max Sources:** {max_sources}

**Focus Areas:**
{focus_str}

**Output Format:** {output_format}

Conduct the research and return your findings in the specified format."""

        # Start with clean state: just the structured prompt and shared files
        subagent_state = {
            "messages": [HumanMessage(content=prompt)],
        }
        # Optionally pass files if they exist in parent state
        files = runtime.state.get("files")
        if files:
            subagent_state["files"] = files

        result = subagent_graph.invoke(subagent_state)

        return Command(
            update={
                "messages": [
                    ToolMessage(
                        result["messages"][-1].text.rstrip(),
                        tool_call_id=runtime.tool_call_id,
                    )
                ],
            }
        )

    return StructuredTool.from_function(
        name="research",
        func=research,
        description='Delegate a structured research task to a specialist subagent.',
    )


print('Structured task tool builder defined')

Structured task tool builder defined


In [14]:
# 创建研究子智能体和编排器

# 子智能体：一个具备搜索能力的简单 create_agent
research_subagent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt='你是研究专家。请严格遵循任务说明。',
    middleware=[
        PatchToolCallsMiddleware(),
    ],
)

# 构建结构化任务工具
structured_research_tool = build_structured_task_tool(research_subagent)

# 编排器：使用结构化任务工具
orchestrator = create_agent(
    model,
    tools=[structured_research_tool],
    system_prompt="""你是研究协调员。当用户要求研究某个主题时：
1. 使用 `research` 工具委托给专家
2. 始终指定 depth、output_format 和相关的 focus_areas
3. 为用户总结研究结果""",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        PatchToolCallsMiddleware(),
    ],
).with_config({"recursion_limit": 100})

print('编排器已准备好结构化研究工具')

Orchestrator ready with structured research tool


In [15]:
# Test: The orchestrator now passes structured fields, not raw strings
config = {"configurable": {"thread_id": str(uuid7())}}

result = orchestrator.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "研究 LangGraph 在 2025 年的新变化。重点关注智能体框架和中间件系统，并用要点形式回答。",
            }
        ]
    },
    config=config,
)

print(result["messages"][-1].content[:2000])

Here's what's new with LangGraph in 2025 regarding the agent framework and middleware system in bullet points:

- Agent Framework Updates:
  - Focus on the core agent loop for improved control and flexibility.
  - Introduction of a new middleware concept for custom hooks at different stages of the agent loop.
  - Support for the latest model integrations and diverse content types.
  - Standard tool calling architecture with provider-agnostic design.
  - Provides both high-level abstractions for common agent patterns and fine-grained control for complex workflows.
  - Emphasis on stateful workflows, allowing agents to remember interaction context.
  - Graph-based architecture enables visual, intuitive workflow designs.

- Middleware System Enhancements:
  - Middleware allows customization within the agent loop, including context compression, summary generation, and message processing enhancements.
  - Customizable middleware like SummarizationMiddleware to exclude specific outputs (e.g.

### 要点
- 默认的 `task()` 工具将原始字符串传递为 `HumanMessage` — 您可以将其替换为结构化 Pydantic 字段
- 自定义任务工具使您可以完全控制子代理接收的状态

---
## 第 5 部分：StateGraph 工作流作为子代理

子代理不必是代理（LLM 循环）。您可以插入任何输出中具有 `messages` 的 `Runnable` — 包括确定性 `StateGraph` 工作流程。

当您有一个不应该留给法学硕士来编排的**固定管道**（收集→分析→格式）时，这是非常强大的。

In [16]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# Define the workflow state — must include  `messages`  for CompiledSubAgent
class ResearchWorkflowState(TypedDict):
    messages: list  # Required: CompiledSubAgent reads the last message
    topic: str  # Extracted from the incoming HumanMessage
    raw_data: str  # Output of the gather step
    analysis: str  # Output of the analyze step

    # Step 1: Extract the topic and gather data


def gather_data(state: ResearchWorkflowState):
    """Search for information on the topic."""
    # The task tool sends HumanMessage(description) -- extract the topic
    topic = state["messages"][-1].content

    # Use tavily directly (not as a tool -- this is a deterministic step)
    results = tavily_client.search(topic, max_results=3)
    raw = "\n\n".join(f"Source: {r['url']}\n{r['content']}" for r in results["results"])
    return {"topic": topic, "raw_data": raw}

    # Step 2: LLM analysis of the raw data


def analyze_data(state: ResearchWorkflowState):
    """Have the LLM analyze and synthesize the gathered data."""
    response = model.invoke(
        [
            {
                "role": "system",
                "content": 'Analyze the following research data. Identify key themes, facts, and insights. Be concise.',
            },
            {
                "role": "user",
                "content": f"Topic: {state['topic']}\n\nData:\n{state['raw_data']}",
            },
        ]
    )
    return {"analysis": response.content}

    # Step 3: Format the final report and set messages for the parent agent


def format_report(state: ResearchWorkflowState):
    """Format into a final report and return via messages."""
    response = model.invoke(
        [
            {
                "role": "system",
                "content": 'Write a concise research summary with bullet points based on this analysis.',
            },
            {"role": "user", "content": state["analysis"]},
        ]
    )
    # The last message is what gets returned to the parent agent
    return {"messages": [AIMessage(content=response.content)]}

    # Build the workflow graph


workflow = StateGraph(ResearchWorkflowState)
workflow.add_node("gather", gather_data)
workflow.add_node("analyze", analyze_data)
workflow.add_node("format", format_report)
workflow.add_edge(START, "gather")
workflow.add_edge("gather", "analyze")
workflow.add_edge("analyze", "format")
workflow.add_edge("format", END)

compiled_workflow = workflow.compile()
print('研究工作流已编译：收集 → 分析 → 格式化')

Research workflow compiled: gather → analyze → format


In [17]:
# Test the workflow standalone first
standalone_result = compiled_workflow.invoke(
    {"messages": [HumanMessage(content='Latest developments in AI agents 2025')]}
)

print('Workflow output (last message):')
print(standalone_result["messages"][-1].content[:1000])

Workflow output (last message):
**Research Summary: AI Agents in 2025**

- **Multi-Agent Collaboration**: Emergence of platforms like "Agentic Studio" featuring 20 AI-powered agents working collectively, highlighting a trend toward integrated multi-agent systems.

- **Business Transformation**: Widespread adoption of AI agents such as Claude, Amelia, and North to automate various business functions, improving efficiency and reducing time expenditures.

- **Technological Advancements**:
  - Multimodal Capabilities: AI agents increasingly handle diverse data types (e.g., text, images) for enhanced interaction.
  - From Single to Multi-Agent Systems: Shift toward collaborative agent frameworks enabling complex problem-solving.
  - Enhanced Reasoning: Improved AI reasoning abilities facilitate more informed and autonomous decision-making.

- **Overall Impact**: 2025 is characterized by significant progress in collaborative, multimodal, and reasoning-capable AI agents that accelerate innova

In [18]:
# Now plug it in as a CompiledSubAgent
from deepagents import create_deep_agent

agent_with_workflow = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt='你是研究协调员。请使用 research-pipeline 进行结构化研究。',
    subagents=[
        {
            # CompiledSubAgent: any Runnable with 'messages' in state
            "name": "research-pipeline",
            "description": 'A deterministic research pipeline that gathers data, analyzes it, and produces a structured report. Use this for any research request.',
            "runnable": compiled_workflow,
        },
    ],
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_workflow.invoke(
    {"messages": [{"role": "user", "content": '研究LangGraph中间件系统的现状'}]},
    config=config,
)

print(result["messages"][-1].content[:2000])

LangGraph's middleware system is currently in a consolidation phase focusing on stability and enhanced type safety. Its architecture remains centered on an agent-based runtime environment with mature core graph APIs and execution models. Recent updates (v1 release) prioritize system robustness and bug fixes without major new features or architectural changes. Comprehensive official documentation and active ecosystem maintenance support developers, reflecting a stable and reliable middleware platform poised for future growth and enhancements.


### 代理子代理与工作流子代理

| |代理子代理（默认）|工作流程子代理 |
|---|---|---|
| **执行** | LLM决定步骤|固定管道|
| **灵活性** |可以即时适应 |确定性|
| **成本** |更多代币（LLM推理） |更少的代币（结构化步骤）|
| **调试** |更难（法学硕士的选择各不相同）|更容易（固定路径）|
| **何时使用** |开放式任务 |明确的管道 |

`CompiledSubAgent` 合约很简单：你的 `Runnable` 的输出状态必须有 `messages` 。最后一条消息作为 `ToolMessage` 返回给父级。

### 要点
- 任何处于 `messages` 状态的 `Runnable` 都可以通过 `CompiledSubAgent` 成为子代理 
- 确定性工作流程比代理子代理更便宜且更可预测
- 混合两者：将代理子代理用于开放式任务，将工作流子代理用于管道

---
## 第 6 部分：用于工具的 MCP 适配器

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/) 提供了一种标准方式，用来暴露来自外部服务器的工具。关键点是：**MCP 工具会变成普通的 LangChain `BaseTool` 对象**，因此它们可以和其他工具互换使用。

### MCP 工具适配器如何工作

```text
MCP Server -> langchain-mcp-adapters -> LangChain BaseTool -> Agent
```

1. **使用 `FastMCP` 编写 MCP 服务器**（或使用已有服务器）
2. **使用 `langchain-mcp-adapters` 通过 stdio 或 HTTP/SSE 连接**
3. **通过 `load_mcp_tools()` 加载工具**，这些工具会变成标准 `BaseTool` 对象
4. **像使用其他 LangChain 工具一样使用它们**

让我们使用带有电子邮件工具的 MCP 服务器构建一个真实示例。


In [19]:
# 我们的 MCP 服务器位于 mcp/email_tools.py — 一个带有 3 个工具的 FastMCP 服务器：
# 写电子邮件、检查日历可用性、安排会议
# 它通过 stdio 传输运行（作为子进程生成）。
#
# 让我们连接到它并加载工具。

import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

mcp_server_path = Path().resolve().parent.parent / "mcp" / 'email_tools.py'
server_params = StdioServerParameters(
    command=sys.executable,  # 相同的 Python 解释器（尊重 venv）
    args=["-u", str(mcp_server_path)],  # -u = 无缓冲的标准输出（stdio 必需）
)


async def demo_mcp_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # load_mcp_tools 转换 MCP 工具 → 标准 LangChain BaseTool 对象
            tools = await load_mcp_tools(session)

            print(f"Loaded {len(tools)} MCP tools:\n")
            for tool in tools:
                print(f"  {tool.name}: {tool.description}")
                print(f"    Type: {type(tool).__name__}")
                print()

                # 直接打电话证明它有效
            result = await tools[0].ainvoke(
                {"to": 'alice@example.com', "subject": "Hello", "content": '嗨爱丽丝！'}
            )
            print(f"直接工具调用结果: {result}")


await demo_mcp_tools()

Loaded 3 MCP tools:

  write_email: Write and send an email.
    Type: StructuredTool

  check_calendar_availability: Check calendar availability for a given day.
    Type: StructuredTool

  schedule_meeting: Schedule a calendar meeting.
    Type: StructuredTool

Direct tool call result: [{'type': 'text', 'text': "Email sent to alice@example.com with subject 'Hello'", 'id': 'lc_3aa32309-7c34-4204-ac27-7df99eb90558'}]


In [20]:
# 将 MCP 工具与 create_agent 结合使用 — 它们是标准 BaseTool 对象


async def demo_agent_with_mcp():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)

            # 将 MCP 工具与常规工具混合使用
            all_tools = [tavily_search, *tools]

            agent = create_agent(
                model,
                tools=all_tools,
                system_prompt='您是一位具有电子邮件和日历功能的有用助手。',
                middleware=[PatchToolCallsMiddleware()],
            )

            config = {"configurable": {"thread_id": str(uuid7())}}
            result = await agent.ainvoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": '检查我星期一的日历是否有空，并向 bob@example.com 发送电子邮件，让他知道我什么时候有空。',
                        }
                    ]
                },
                config=config,
            )

            print(result["messages"][-1].content)


await demo_agent_with_mcp()

I checked your calendar and you are available on Monday at 9:00 AM, 2:00 PM, and 4:00 PM. I have sent an email to Bob letting him know your available times. Is there anything else you would like to do?


### 当 MCP 更改您的调用模式时

MCP 工具是**远程过程调用**。这具有实际意义：

|方面|本机 `@tool` | MCP 工具 |
|--------------------|----------------|----------|
|延迟 |进行中|网络往返（子进程或 HTTP）|
|国家准入|  `ToolRuntime` 可用 |无状态访问（无状态 RPC）|
|错误处理 | Python 异常 | MCP错误协议|
|生命周期|与经纪人同住|服务器进程必须正在运行 |

最重要的区别：**MCP 工具无法访问 `ToolRuntime` **。它们无法读取代理状态、返回 `Command` 对象或直接修改状态。它们是纯粹的输入→输出函数。

这意味着您无法用 MCP 文件系统工具替换 `FilesystemMiddleware` 工具并获得相同的状态管理。 MCP 工具最适合**外部功能**（电子邮件、日历、数据库、API），而不是状态耦合操作。### 要点
- MCP 服务器通过 `FastMCP` 公开工具 — 编写一次，从任何 MCP 客户端使用
- `load_mcp_tools()` 将 MCP 工具转换为标准 `BaseTool` 对象
- 自由混合 MCP 和本机工具 — 代理不知道区别
- MCP 工具是无状态 RPC — 它们无法访问代理状态或返回 `Command` 对象

---
## 第 7 部分：沙箱使用

当后端实现 `SandboxBackendProtocol` 时，深度代理可以执行 shell 命令。这会向代理添加一个 `execute` 工具。

### 沙箱后端层次结构

 ```
BackendProtocol               <- File operations only (ls, read, write, edit, glob, grep)
  └── SandboxBackendProtocol  <- + execute(command, timeout) + id property
        └── BaseSandbox       <- Implements ALL file ops as shell commands via execute()
              └── LangSmithSandbox   <- Wraps langsmith.sandbox.SandboxClient
              └── YourOwnSandbox     <- Just implement execute()!
``` 

 `BaseSandbox` 通过 `execute()` 脱壳命令来实现**每个**文件操作（ `read` 、 `write` 、 `edit` 、 `ls` 、 `glob` 、 `grep` ）。因此，要构建自己的沙箱后端，您只需要实现**一个方法**： `execute()` 。

### LangSmith 沙箱 SDK

 ```python
from langsmith.sandbox import SandboxClient

client = SandboxClient()  # Uses LANGSMITH_API_KEY
client.create_template(name="my-sandbox", image="python:3.12-slim")

with client.sandbox(template_name="my-sandbox") as sb:
    sb.write("/app/script.py", "print('hello')")
    result = sb.run("python /app/script.py")
    print(result.stdout)  # "hello\n"
``` 

Deep Agents 用 `LangSmithSandbox(sb)` 包装它，将其公开为 `SandboxBackendProtocol` 。

In [21]:
# 控制沙箱功能的协议
from deepagents.backends.protocol import (
    BackendProtocol,
    SandboxBackendProtocol,
    ExecuteResponse,
)

# BackendProtocol方法（文件操作）：
backend_methods = [
    m for m in dir(BackendProtocol) if not m.startswith('_') and not m.startswith('a')
]
print('后端协议方法：', backend_methods)

# SandboxBackendProtocol 添加：
sandbox_only = [
    m
    for m in dir(SandboxBackendProtocol)
    if not m.startswith('_') and not m.startswith('a') and m not in dir(BackendProtocol)
]
print('SandboxBackendProtocol 添加：', sandbox_only)

print(f"\nExecuteResponse fields: output (str), exit_code (int|None), truncated (bool)")

BackendProtocol methods: ['download_files', 'edit', 'glob_info', 'grep_raw', 'ls_info', 'read', 'upload_files', 'write']
SandboxBackendProtocol adds: ['execute', 'id']

ExecuteResponse fields: output (str), exit_code (int|None), truncated (bool)


In [22]:
# （可选）运行示例：LangSmith Sandbox
# 需要 LANGSMITH_API_KEY 以及支持沙箱的计划。
# 如果您无权访问，请跳过此单元格 - 下面的概念部分涵盖了该模式。

import os

if os.environ.get("LANGSMITH_API_KEY"):
    try:
        from langsmith.sandbox import SandboxClient
        from deepagents.backends.langsmith import LangSmithSandbox

        client = SandboxClient()

        # 创建一个模板（仅需要一次 - 可以安全地再次调用）
        try:
            client.create_template(name="deepagents-extra", image="python:3.12-slim")
            print('创建模板“deepagents-extra”')
        except Exception:
            print('模板“deepagents-extra”已存在')

            # 启动沙箱并运行命令
        with client.sandbox(template_name="deepagents-extra") as sb:
            # 直接使用SDK
            result = sb.run("echo '来自沙箱的你好！' && python3 --版本")
            print(f"\nDirect execution:")
            print(f"  stdout: {result.stdout.strip()}")
            print(f"  success: {result.success}")

            # 文件操作
            sb.write('/应用程序/hello.py', "print('来自沙盒 Python 脚本的问候！')")
            result = sb.run('python3 /app/hello.py')
            print(f"\nScript execution:")
            print(f"  stdout: {result.stdout.strip()}")

            # 包装为 Deep Agents 后端并与 create_deep_agent 一起使用
            backend = LangSmithSandbox(sb)
            print(f"\nSandbox ID: {backend.id}")
            print(f"Backend type: {type(backend).__name__}")
            print(
                f"Is SandboxBackendProtocol: {isinstance(backend, SandboxBackendProtocol)}"
            )

            sandbox_agent = create_deep_agent(
                model=model,
                backend=backend,
                system_prompt='你是一名编码助理。在沙箱中编写并运行代码。',
            )

            config = {"configurable": {"thread_id": str(uuid7())}}
            result = sandbox_agent.invoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": '向 /app/fib.py 编写一个 Python 单行代码，打印前 10 个斐波那契数，然后运行它。',
                        }
                    ]
                },
                config=config,
            )
            print(f"\nAgent response: {result['messages'][-1].content}")

    except Exception as e:
        print(f"Sandbox error: {e}")
        print('沙箱需要有支持它们的 LangSmith 计划。')
else:
    print('LANGSMITH_API_KEY 未设置 — 跳过 LangSmith 沙箱演示。')
    print('在 .env 中设置它以运行该单元。')

/var/folders/jj/2fvdkyfj0856p6_6sdvv74rw0000gn/T/ipykernel_85201/1028452110.py:9: FutureWarning: langsmith.sandbox is in alpha. This feature is experimental, and breaking changes are expected.
  from langsmith.sandbox import SandboxClient


Created template 'deepagents-extra'

Direct execution:
  stdout: Hello from sandbox!
Python 3.12.13
  success: True

Script execution:
  stdout: Hello from a sandboxed Python script!

Sandbox ID: sandbox-387e75f8
Backend type: LangSmithSandbox
Is SandboxBackendProtocol: True

Agent response: Python one-liner to print the first 10 Fibonacci numbers has been written to /app/fib.py and executed. The output is: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34].


In [23]:
# 构建您自己的沙箱后端
#
# BaseSandbox通过execute()处理所有文件操作。
# 你只需要实现：execute()、id、upload_files()、download_files()
#
# 这是模式（从 langchain-ai/openshell-deepagent 简化）：

from deepagents.backends.sandbox import BaseSandbox
from deepagents.backends.protocol import (
    ExecuteResponse,
    FileUploadResponse,
    FileDownloadResponse,
)
import subprocess


class LocalDockerSandbox(BaseSandbox):
    """示例：在本地 Docker 容器中运行命令。

    在生产中，您将使用真正的沙箱服务（LangSmith、OpenShell、Modal 等）
    这只是演示了该模式。"""

    def __init__(self, container_id: str):
        self._container_id = container_id

    @property
    def id(self) -> str:
        return self._container_id

    def execute(self, command: str, *, timeout: int | None = None) -> ExecuteResponse:
        """在容器内运行命令 - 这是您必须实现的唯一方法。"""
        try:
            result = subprocess.run(
                ["docker", "exec", self._container_id, "bash", "-c", command],
                capture_output=True,
                text=True,
                timeout=timeout or 60,
            )
            output = result.stdout
            if result.stderr:
                output = f"{output}\n{result.stderr}" if output else result.stderr
            return ExecuteResponse(output=output, exit_code=result.returncode)
        except subprocess.TimeoutExpired:
            return ExecuteResponse(output='命令超时', exit_code=1)

            # upload_files 和 download_files 在 BaseSandbox 中有默认实现
            # 使用带有 Base64 编码的execute()。仅当您的沙箱
            # 具有更高效的本机文件传输机制。


print('LocalDockerSandbox 定义 — 演示 BaseSandbox 模式')
print()
print('BaseSandbox 免费为您提供什么（全部通过execute()）：')
print('ls、读取、写入、编辑、glob、grep')
print()
print('您实施的内容：')
print('执行（）——必需')
print('id — 必需（属性）')
print('upload_files() — 可选（默认使用execute + base64）')
print('download_files() — 可选（默认使用execute + base64）')

LocalDockerSandbox defined — demonstrates the BaseSandbox pattern

What BaseSandbox gives you for free (all via execute()):
  ls, read, write, edit, glob, grep

What you implement:
  execute() — required
  id — required (property)
  upload_files() — optional (default uses execute + base64)
  download_files() — optional (default uses execute + base64)


### 使用带有 `create_deep_agent` 的沙箱后端 

**模式1：直接沙箱后端**
 ```python
agent = create_deep_agent(
    model=model,
    backend=LocalDockerSandbox("my-container-id"),
)
``` 

**模式 2：CompositeBackend — 代码沙箱，内存文件系统**
 ```python
from deepagents.backends import CompositeBackend, FilesystemBackend

def create_backend(runtime):
    return CompositeBackend(
        default=LocalDockerSandbox("my-container-id"),
        routes={
            "/memories/": FilesystemBackend(root_dir="./data", virtual_mode=True),
        },
    )

agent = create_deep_agent(model=model, backend=create_backend)
``` 

代理会自动获取 `execute` 工具，因为后端实现了 `SandboxBackendProtocol` 。

### 要点
- 要构建您自己的沙箱后端，子类 `BaseSandbox` 并实现 `execute()` — 就是这样
- `BaseSandbox` 通过 shell 命令从 `execute()` 派生所有文件操作（ `ls` 、 `read` 、 `write` 等）
- 使用 `CompositeBackend` 将沙箱（用于代码执行）与 `FilesystemBackend` （用于持久内存/技能）混合
- `LangSmithSandbox` 是生产就绪的实现；请参阅 [openshell-deepagent]( https://github.com/langchain-ai/openshell-deepagent) 获取另一个示例

---
## 总结

|部分|你学到了什么 |关键图案|
|------|-----------------|-------------|
| **1.解构** |  `create_deep_agent` = `create_agent` + 中间件 |中间件堆栈组成 |
| **2.文件系统** |工具返回 `Command(update={...})` 来修改状态 |状态减速器+命令模式|
| **3.后端** |  `WriteResult.files_update` 控制状态与直接持久性 |后端抽象，`CompositeBackend` |
| **4.任务定义** |用结构化 Pydantic 输入替换原始字符串任务 |每个子代理自定义 `StructuredTool` |
| **5.工作流子代理** |任何带有 `messages` 的 `Runnable` 都可以是子代理 |  `CompiledSubAgent` 合同 |
| **6。 MCP 适配器** | MCP 工具成为标准 `BaseTool` 对象 |适配器模式、无状态RPC |
| **7.沙箱** |  `SandboxBackendProtocol` 为 `execute` 工具提供门控 |协议扩展模式|

### 从这里到哪里去- **构建您自己的中间件** - 现在您已经了解了挂钩，可以创建用于日志记录、缓存、速率限制或自定义行为的中间件
- **自定义后端** — 为您的存储（S3、数据库等）实施 `BackendProtocol`
- **使用 LangGraph Platform 进行部署** — 使用 `langgraph dev` 通过 Studio UI 运行代理
- **评估** — 使用 LangSmith 通过数据集和评估器来衡量代理质量